# Scale OCR — YOLOv8s Training on GPU
**Runtime → Change runtime type → T4 GPU**

In [ ]:
# 1. Install dependencies
!pip install -q ultralytics roboflow

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None')

In [ ]:
# 2. Download dataset from Roboflow
from roboflow import Roboflow
rf = Roboflow(api_key="YOUR_ROBOFLOW_API_KEY")  # replace with your key
project = rf.workspace("rishis-workspace-eyqrq").project("digital-weight-scale-detection-2")
version = project.version(1)
dataset = version.download("yolov8")
DATA_YAML = dataset.location + "/data.yaml"
print('Dataset at:', DATA_YAML)

In [ ]:
# 3. Augment negative sign samples (only 2 in dataset)
import cv2, numpy as np
from pathlib import Path

TRAIN_IMG = Path(dataset.location) / 'train' / 'images'
TRAIN_LBL = Path(dataset.location) / 'train' / 'labels'

# Find negative sign files
neg_files = []
for lbl in TRAIN_LBL.glob('*.txt'):
    if any(line.startswith('0 ') for line in lbl.read_text().splitlines()):
        neg_files.append(lbl.stem)
print(f'Found {len(neg_files)} negative sign files: {neg_files}')

created = 0
for stem in neg_files:
    img = cv2.imread(str(TRAIN_IMG / f'{stem}.jpg'))
    lbl_text = (TRAIN_LBL / f'{stem}.txt').read_text()
    variants = []
    for alpha, beta in [(0.7,-20),(0.85,-10),(1.15,10),(1.3,20),(1.5,40),(1.7,60)]:
        variants.append(cv2.convertScaleAbs(img, alpha=alpha, beta=beta))
    for angle in [-3,-2,-1,1,2,3]:
        h,w = img.shape[:2]
        M = cv2.getRotationMatrix2D((w/2,h/2), angle, 1.0)
        variants.append(cv2.warpAffine(img, M, (w,h)))
    for sigma in [3,5,8]:
        noise = np.random.normal(0,sigma,img.shape).astype(np.int16)
        variants.append(np.clip(img.astype(np.int16)+noise,0,255).astype(np.uint8))
    for i, aug in enumerate(variants):
        cv2.imwrite(str(TRAIN_IMG / f'{stem}_aug{i:02d}.jpg'), aug)
        (TRAIN_LBL / f'{stem}_aug{i:02d}.txt').write_text(lbl_text)
        created += 1

print(f'Created {created} augmented negative samples')
print(f'Total train images: {len(list(TRAIN_IMG.glob("*.jpg")))}')

In [ ]:
# 4. Train YOLOv8s at 1280px on GPU
from ultralytics import YOLO

model = YOLO('yolov8s.pt')
model.train(
    data=DATA_YAML,
    epochs=300,
    imgsz=1280,
    batch=8,          # GPU can handle larger batch
    device=0,         # GPU
    workers=4,
    patience=60,
    project='runs/detect',
    name='scale_ocr_v3',
    exist_ok=True,
    # Augmentation
    fliplr=0.0,
    flipud=0.0,
    degrees=3.0,
    translate=0.05,
    scale=0.2,
    shear=1.0,
    hsv_h=0.01,
    hsv_s=0.4,
    hsv_v=0.5,
    mosaic=0.0,
    mixup=0.0,
    erasing=0.0,
    copy_paste=0.0,
    # Optimiser
    optimizer='AdamW',
    lr0=0.0005,
    lrf=0.01,
    weight_decay=0.0005,
    warmup_epochs=5,
    cos_lr=True,
    label_smoothing=0.05,
    # Loss
    box=5.0,
    cls=2.0,
    dfl=1.5,
)

In [ ]:
# 5. Check results
import pandas as pd
results = pd.read_csv('runs/detect/scale_ocr_v3/results.csv')
best_map = results['metrics/mAP50(B)'].max()
print(f'Best mAP50: {best_map:.4f}')
print(results[['epoch','metrics/mAP50(B)','metrics/precision(B)','metrics/recall(B)']].tail(10).to_string())

In [ ]:
# 6. Download best.pt
from google.colab import files
files.download('runs/detect/scale_ocr_v3/weights/best.pt')